# 05 — Memory / stateful agent

**Definition:** the agent keeps state **across** runs — it has identity over time.

Framing that took me a while: memory is usually an **augmentation**, not a standalone
architecture. A ReAct agent + memory is a memory-enabled ReAct agent. It's a capability I
bolt onto patterns 1–4, 6 and 7.

There are three layers, and mixing them up is the source of most confusion:

```
+------------------------------------------------------+
| 1. WITHIN-RUN            state["messages"]           |
|    lives: one invoke()                               |
|    "what did the tool just return?"                  |
+------------------------------------------------------+
| 2. SHORT-TERM (thread)   CHECKPOINTER                |
|    lives: one conversation (thread_id)               |
|    "what did the user say 3 turns ago?"              |
+------------------------------------------------------+
| 3. LONG-TERM (cross-thread)  STORE                   |
|    lives: forever, scoped by user_id                 |
|    "this user is vegetarian and lives in Cologne"    |
+------------------------------------------------------+
```

**Checkpointer vs Store** — the table I keep coming back to:

| | checkpointer | store |
|---|---|---|
| scope | one `thread_id` | across all threads |
| stores | the **whole graph state** | app-defined **JSON documents** |
| keyed by | `thread_id` | a namespace tuple, e.g. `("memories", user_id)` |
| written by | **LangGraph, automatically** | **me, explicitly, in a node** |
| analogy | the session | the user profile |
| dev / prod | `InMemorySaver` / `PostgresSaver` | `InMemoryStore` / `PostgresStore` |
| gives you | a **conversation** | a **relationship** |

You almost always need both. Passing only one is the classic architecture mistake.

## Setup

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

# Both providers serve the SAME model (gpt-oss-120b), so behaviour is identical.
# Groq is the default; set LLM_PROVIDER=cerebras to switch. I added that second path
# after burning through Groq's 200k-tokens-per-day cap while writing these notebooks.
if os.environ.get("LLM_PROVIDER", "groq") == "cerebras":
    from langchain_openai import ChatOpenAI

    if not os.environ.get("CEREBRAS_API_KEY"):
        os.environ["CEREBRAS_API_KEY"] = getpass.getpass("CEREBRAS_API_KEY: ")
    llm = ChatOpenAI(
        model="gpt-oss-120b",
        temperature=0,
        base_url="https://api.cerebras.ai/v1",
        api_key=os.environ["CEREBRAS_API_KEY"],
        timeout=120,      # ChatOpenAI defaults to NO timeout - a stalled
        max_retries=5,    # connection hangs the whole notebook forever
    )
else:
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
    # reasoning_format="hidden" keeps gpt-oss's chain-of-thought out of .content
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        reasoning_format="hidden",
        timeout=120,
        max_retries=5,
    )


def show(graph):
    print(graph.get_graph().draw_mermaid())


print(llm.invoke("Reply with the single word: ready").content)

# Part A — short-term memory (checkpointer)

Three things and you're done. Note that the node itself is *not* memory-aware at all — the
checkpointer works outside it.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph


def chat_node(state: MessagesState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}


b = StateGraph(MessagesState)
b.add_node("chat", chat_node)
b.add_edge(START, "chat")
b.add_edge("chat", END)

checkpointer = InMemorySaver()                        # (1) create it
chat_graph = b.compile(checkpointer=checkpointer)     # (2) pass it to compile()
config = {"configurable": {"thread_id": "conv-1"}}    # (3) ALWAYS pass a thread_id

r1 = chat_graph.invoke({"messages": [HumanMessage("Hi, I'm Ankit and I live in Cologne.")]}, config)
print("A:", r1["messages"][-1].content[:120])

# Turn 2 - I send ONLY the new message; history is restored from the checkpoint.
r2 = chat_graph.invoke({"messages": [HumanMessage("Where do I live?")]}, config)
print("A:", r2["messages"][-1].content[:120])

print(f"\nmessages now in state: {len(r2['messages'])}  <- history accumulated")

The bit worth internalising: I passed one message, but state holds four. LangGraph loaded
the previous ones from the checkpoint and `add_messages` appended mine:

```
graph.invoke({"messages": [HumanMessage("Where do I live?")]}, config)
                                  |
      [old msgs from checkpoint] + [my new msg]   <- what the LLM actually sees
```

### Threads are isolated — proof

In [ ]:
other = {"configurable": {"thread_id": "conv-2"}}     # a DIFFERENT thread
r = chat_graph.invoke({"messages": [HumanMessage("Where do I live?")]}, other)
print("thread conv-2:", r["messages"][-1].content[:200])
print("\n^ no idea. Different thread_id == different conversation.")

## Inspecting and time-travelling

The checkpointer saves a snapshot **after every node**, which is a debugging superpower:
I can read the state at any point, and re-run from any point.

In [ ]:
snap = chat_graph.get_state(config)
print("messages in thread:", len(snap.values["messages"]))
print("next node to run  :", snap.next, "(empty = finished)")

history = list(chat_graph.get_state_history(config))
print(f"\ncheckpoints saved: {len(history)}")
for h in history[:4]:
    print(f"  {h.config['configurable']['checkpoint_id'][:8]}... "
          f"| msgs={len(h.values.get('messages', []))} | next={h.next}")

### Time travel — resume from an old checkpoint

Pass an old `checkpoint_id` and the graph forks reality from that point. The original
thread is untouched.

In [ ]:
old = history[2]              # some earlier state
fork_cfg = old.config         # this config carries the checkpoint_id
out = chat_graph.invoke({"messages": [HumanMessage("What's my name?")]}, fork_cfg)
print("forked branch says:", out["messages"][-1].content[:150])
print("\noriginal thread still has:", len(chat_graph.get_state(config).values["messages"]), "messages")

### Production checkpointers

| backend | import | for |
|---|---|---|
| `InMemorySaver` | `langgraph.checkpoint.memory` | notebooks, tests |
| `SqliteSaver` | `langgraph.checkpoint.sqlite` | single-process prototypes |
| `PostgresSaver` | `langgraph.checkpoint.postgres` | production |
| `AsyncPostgresSaver` | same | async production |

```python
# prototype (survives restarts, single process only - file locking)
from langgraph.checkpoint.sqlite import SqliteSaver
with SqliteSaver.from_conn_string("./memory.db") as cp:
    graph = builder.compile(checkpointer=cp)

# production
from langgraph.checkpoint.postgres import PostgresSaver
with PostgresSaver.from_conn_string("postgresql://...") as cp:
    cp.setup()                 # run ONCE to create the tables
    graph = builder.compile(checkpointer=cp)
```

`thread_id` must be under 255 chars — Postgres stores it in a bounded column. Use a UUID.

# Part B — long-term memory (store)

Now the part **I** have to write explicitly. Nothing here happens automatically.

In [ ]:
import uuid
from typing import List

from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from pydantic import BaseModel, Field


class Memories(BaseModel):
    """Facts worth persisting about a user."""

    facts: List[str] = Field(
        description="Durable facts about the user (preferences, location, constraints). "
        "Empty list if the message contains nothing worth remembering."
    )


extractor = llm.with_structured_output(Memories, method="json_schema")

### Namespace vs key

Think directory vs filename:

```python
store.put(("memories", "user-42"), "diet", {"value": "vegetarian"})
#          \______ namespace ____/  \key/   \______ value _______/
```

The key choice decides the semantics, and it bites later if you get it wrong:

| data | key strategy | why |
|---|---|---|
| attributes that **change** (diet, city, plan) | deterministic (`"diet"`) | the new value overwrites the old |
| records that **accumulate** (past issues, notes) | UUID | each one is a distinct record |

### How does `store` get into my node? I never passed it.

Dependency injection by **parameter name**. Declare a parameter literally called `store`
and LangGraph injects whatever went into `compile(store=...)`. Same for `config`. The names
are not arbitrary — rename it to `memory_store` and you get nothing.

In [ ]:
def load_memories(state: MessagesState, config: RunnableConfig, *, store: BaseStore) -> dict:
    """Read this user's long-term facts and inject them as a SystemMessage."""
    user_id = config["configurable"].get("user_id", "anonymous")
    namespace = ("memories", user_id)

    # .search() returns Items. With an embedding index, `query=` is a real vector search.
    items = store.search(namespace, query=state["messages"][-1].content, limit=10)

    if not items:
        return {}                      # nothing to inject -> return an EMPTY update

    facts = "\n".join(f"- {i.value['text']}" for i in items)
    print(f"  loaded {len(items)} memories for {user_id}")
    return {"messages": [SystemMessage(content=f"What you know about this user:\n{facts}")]}


def memory_agent(state: MessagesState) -> dict:
    """An ordinary agent. It just happens to find memories in its message list."""
    return {"messages": [llm.invoke(state["messages"])]}


def save_memories(state: MessagesState, config: RunnableConfig, *, store: BaseStore) -> dict:
    """Extract durable facts from the recent turns and persist them."""
    user_id = config["configurable"].get("user_id", "anonymous")
    namespace = ("memories", user_id)

    convo = "\n".join(f"{m.type}: {m.content}" for m in state["messages"][-4:])
    result = extractor.invoke(
        "Extract durable facts about the USER from this exchange. "
        f"Ignore small talk and anything transient.\n\n{convo}"
    )

    for fact in result.facts:
        # UUID key -> facts accumulate rather than overwrite each other
        store.put(namespace, str(uuid.uuid4()), {"text": fact})
        print(f"  saved: {fact}")

    return {}    # the write went to the STORE, not to state -> no state update

### Wire it up — **both** checkpointer and store

In [ ]:
mb = StateGraph(MessagesState)
mb.add_node("load", load_memories)
mb.add_node("agent", memory_agent)
mb.add_node("save", save_memories)

mb.add_edge(START, "load")
mb.add_edge("load", "agent")
mb.add_edge("agent", "save")
mb.add_edge("save", END)

store = InMemoryStore()
saver = InMemorySaver()

# checkpointer = session continuity. store = relationship continuity.
mem_graph = mb.compile(checkpointer=saver, store=store)
show(mem_graph)

## The payoff — a brand-new thread that still knows me

In [ ]:
cfg1 = {"configurable": {"thread_id": "session-1", "user_id": "ankit"}}
print("SESSION 1")
r = mem_graph.invoke(
    {
        "messages": [
            HumanMessage(
                "I'm a backend engineer in Cologne. I'm vegetarian and I'm training "
                "for a sub-60-minute 10K."
            )
        ]
    },
    cfg1,
)
print("A:", r["messages"][-1].content[:200])

In [ ]:
# New thread_id, same user_id. The CHECKPOINTER has nothing to offer here.
cfg2 = {"configurable": {"thread_id": "session-2", "user_id": "ankit"}}
print("SESSION 2 (new thread)")
r = mem_graph.invoke({"messages": [HumanMessage("Suggest a dinner for tonight.")]}, cfg2)
print("A:", r["messages"][-1].content[:300])
print("\n^ it knows I'm vegetarian. That came from the STORE, not the checkpointer.")

In [ ]:
print("stored memories for 'ankit':")
for item in store.search(("memories", "ankit")):
    print(f"  [{item.key[:8]}] {item.value['text']}")

print("\nmemories for a different user:", store.search(("memories", "someone-else")))
print("^ namespace isolation: users cannot see each other's facts.")

### Semantic search in the store

`InMemoryStore` does substring matching by default. Add an embedding index and `query=`
becomes a real vector search:

```python
from langchain_openai import OpenAIEmbeddings

semantic_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(model="text-embedding-3-small"),
        "dims": 1536,
        "fields": ["text"],        # which value keys get embedded
    }
)
semantic_store.search(("memories", "ankit"), query="food restrictions", limit=3)
```

Not run here — it needs an embeddings provider. In production it's `PostgresStore`
(pgvector); the API is identical, only the import and constructor change.

# Part C — the short version, and keeping context from exploding

## Memory on a prebuilt agent

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def log_run(distance_km: float, minutes: int) -> str:
    """Record a completed run."""
    return f"Logged {distance_km}km in {minutes}min ({minutes / distance_km:.1f} min/km)"


smart_agent = create_agent(
    model=llm,
    tools=[log_run],
    system_prompt="You are a running coach.",
    checkpointer=InMemorySaver(),    # short-term
    store=InMemoryStore(),           # long-term (tools can reach it)
)

c = {"configurable": {"thread_id": "run-1", "user_id": "ankit"}}
smart_agent.invoke({"messages": [("user", "I ran 5km in 28 minutes today.")]}, c)
out = smart_agent.invoke({"messages": [("user", "How far did I run today?")]}, c)
print(out["messages"][-1].content)

## Context management — set this up *before* you need it

At ~50 turns the context window becomes the problem. Three strategies:

| strategy | how | trade-off |
|---|---|---|
| trim | `trim_messages(max_tokens=...)` | cheap, hard-loses old info |
| summarise | a node that condenses old turns into one message | keeps the gist, costs a call |
| extract to store | pull facts out, drop the raw turns | best long-term, most work |

### Trimming

In [ ]:
from langchain_core.messages import trim_messages

long_history = [
    SystemMessage("You are a running coach."),
    HumanMessage("I ran 5km today."),
    *[HumanMessage(f"Filler message number {i}, padding out the history.") for i in range(40)],
    HumanMessage("What should I run tomorrow?"),
]

trimmed = trim_messages(
    long_history,
    strategy="last",          # keep the most recent
    token_counter=len,        # count MESSAGES, not tokens (see the note below)
    max_tokens=6,             # so this means "keep 6 messages"
    start_on="human",         # never start on a ToolMessage - that's an API error
    include_system=True,      # always keep the system prompt
)

print(f"{len(long_history)} messages -> {len(trimmed)} after trimming")
for m in trimmed:
    print(f"  {m.type}: {m.content[:60]}")

**Gotcha:** the docs mostly show `token_counter=llm`, which counts with the model's real
tokenizer. With `ChatGroq` that raised

```
ImportError: Could not import transformers python package.
This is needed in order to calculate get_token_ids.
```

because there's no native token counter for these models, so langchain-core falls back to a
local GPT-2 tokenizer from `transformers`. Options: `pip install transformers` (a big
dependency for a token count), pass a real counter like `tiktoken`, or — as here —
`token_counter=len` to budget in **messages** instead of tokens. Crude, zero dependencies,
and fine when messages are roughly uniform in size.

### Trimming inside an agent

`trim_messages` on its own is just a list operation. To make it run before *every* LLM call,
it goes in a hook, and the trimmed list is what the model sees while full history stays in
the checkpoint.

In [ ]:
def trim_hook(state):
    return {
        "llm_input_messages": trim_messages(
            state["messages"],
            strategy="last",
            token_counter=len,
            max_tokens=6,
            start_on="human",
            include_system=True,
        )
    }


print("`llm_input_messages` overrides what the LLM sees WITHOUT mutating stored state -")
print("full history stays in the checkpoint, only the prompt shrinks.")

### Summarising, and the only way to delete messages

You cannot shrink an `add_messages` list by returning a shorter one — the reducer appends,
so a short list just gets merged in. `RemoveMessage(id=...)` is the deletion signal.

In [ ]:
from langchain_core.messages import RemoveMessage


class SummaryState(MessagesState):
    summary: str


def summarize_node(state: SummaryState) -> dict:
    """Condense old turns into one summary message, then DELETE them."""
    msgs = state["messages"]
    if len(msgs) <= 6:
        return {}

    prev = state.get("summary", "")
    head = (
        f"Previous summary:\n{prev}\n\nExtend it with these new messages:"
        if prev
        else "Summarize this conversation:"
    )
    text = "\n".join(f"{m.type}: {m.content}" for m in msgs[:-4])
    summary = llm.invoke(f"{head}\n\n{text}").content

    # RemoveMessage(id=...) tells add_messages to DELETE that message.
    deletions = [RemoveMessage(id=m.id) for m in msgs[:-4]]
    return {"summary": summary, "messages": deletions}


sb = StateGraph(SummaryState)
sb.add_node("summarize", summarize_node)
sb.add_edge(START, "summarize")
sb.add_edge("summarize", END)
summary_graph = sb.compile()

res = summary_graph.invoke({"messages": long_history[:12], "summary": ""})
print(f"12 messages in -> {len(res['messages'])} left in state")
print("\nsummary:", res["summary"][:300])

## Notes to self

**"I added a checkpointer but it still forgets."** Always one of three things:

1. no `config={"configurable": {"thread_id": ...}}` — without it you get an error or a
   fresh thread every call
2. a **new** `thread_id` each turn — every one is a separate conversation
3. no reducer on the state key, so each node overwrites instead of appending

**"Why pass the message list AND have a checkpointer? Isn't that double?"** No: state is
the working set for the current run; the checkpointer *persists* it between runs. I pass
only the new message and LangGraph rehydrates the rest.

**Failure modes:**

| symptom | cause | fix |
|---|---|---|
| agent forgets mid-conversation | missing/rotating `thread_id` | one stable id per conversation |
| memories leak between users | namespace not scoped by user | `("memories", user_id)` |
| store injection is `None` | parameter not named `store` | `*, store: BaseStore` |
| facts overwrite each other | deterministic key for accumulating data | UUID keys |
| context window blows up | no trimming or summarising | `trim_messages` / a summarise node |
| deleting messages does nothing | returning a shorter list | `RemoveMessage(id=...)` |

**API I used:**

```python
compile(checkpointer=InMemorySaver())          # short-term
{"configurable": {"thread_id": "x"}}           # the conversation key
compile(store=InMemoryStore())                 # long-term
store.put(("memories", uid), key, {"text": ...})
store.search(("memories", uid), query="...")
def node(state, config, *, store: BaseStore)   # injection by parameter name
graph.get_state(config) / graph.get_state_history(config)
trim_messages(...) / RemoveMessage(id=m.id)
```

Next: **06 — Multi-agent**, where one decision maker becomes several.